# Intrusão salina 1D — comparação das condições de contorno

Este notebook reproduz o experimento sintético que compara Neumann homogênea com a condição comutável Neumann–Danckwerts em $x=L$.

A extremidade marítima $x=0$ permanece como reservatório de salinidade prescrita. Em $x=L$, a condição de Danckwerts é ativada somente quando $u(t)<0$:

$$
uC-D\frac{\partial C}{\partial x}=uC_{\mathrm{mont}}.
$$

O caso é sintético e serve para verificação numérica; ele não representa calibração ou previsão hidrodinâmica do Rio São Mateus.

In [ ]:
# Execute esta célula uma vez, em uma sessão nova.
from pathlib import Path
import subprocess
import sys
import zipfile

IN_COLAB = "google.colab" in sys.modules
if IN_COLAB:
    archive = Path("/content/salt_intrusion_1d_v0.4.0.zip")
    if not archive.exists():
        raise FileNotFoundError(
            "Envie salt_intrusion_1d_v0.4.0.zip para /content e execute novamente."
        )
    with zipfile.ZipFile(archive) as compressed:
        compressed.extractall("/content")
    project_dir = Path("/content/salt_intrusion_1d")
else:
    project_dir = Path("..").resolve()

subprocess.check_call(
    [sys.executable, "-m", "pip", "install", "--no-deps", "--force-reinstall", str(project_dir)]
)
%matplotlib inline

In [ ]:
from pathlib import Path

import matplotlib.pyplot as plt

from salt_intrusion_1d.experiment import (
    MOUTH_OFFSET_KM,
    plot_results,
    run_comparison,
    write_cycle_summary,
    write_last_cycle_series,
    write_summary,
)

## Execução

A configuração principal usa $L=50$ km, $\Delta x=100$ m, $\Delta t=60$ s e 60 ciclos de maré. Para um teste rápido, troque `cycles=60` por `cycles=3` e `n_cells=500` por `n_cells=200`.

In [ ]:
results = run_comparison(
    cycles=60,
    n_cells=500,
    dt_s=60.0,
    store_every_steps=30,
)

for (discharge, boundary), result in sorted(results.items()):
    mean_domain = result.mean_intrusion_last_cycle_m() / 1_000
    maximum_domain = result.max_intrusion_last_cycle_m() / 1_000
    print(
        f"Q={discharge:>4.1f} m³/s | {boundary:11s} | "
        f"média={mean_domain + MOUTH_OFFSET_KM:7.3f} km da foz | "
        f"máximo={maximum_domain + MOUTH_OFFSET_KM:7.3f} km da foz"
    )

## Visualização do efeito da fronteira

O gráfico abaixo mostra o último ciclo para $Q=2\,\mathrm{m^3/s}$, cenário em que a frente salina alcança a vizinhança de $x=L$. A condição de Neumann permanente não introduz água de montante durante a reversão. Danckwerts incorpora esse fluxo de entrada e permite a lavagem do domínio.

In [ ]:
discharge = 2.0
labels = {"neumann": "Neumann homogênea", "danckwerts": "Danckwerts"}
styles = {"neumann": "-", "danckwerts": "--"}

reference = results[(discharge, "neumann")]
cycle_start_h = (
    reference.times_s[-1] - reference.config.tidal_period_s
) / 3_600

fig, ax = plt.subplots(figsize=(7.2, 4.4))
for boundary in ("neumann", "danckwerts"):
    result = results[(discharge, boundary)]
    mask = result.last_cycle_mask()
    ax.plot(
        result.times_s[mask] / 3_600 - cycle_start_h,
        result.intrusion_length_m[mask] / 1_000 + MOUTH_OFFSET_KM,
        styles[boundary],
        linewidth=2.2,
        label=labels[boundary],
    )
ax.set_xlabel("Tempo no último ciclo de maré (h)")
ax.set_ylabel("Distância da frente salina à foz (km)")
ax.set_title(r"Comprimento de intrusão: $Q=2\,\mathrm{m^3/s}$")
ax.grid(alpha=0.25)
ax.legend(frameon=False)
plt.show()

## Exportação reproduzível

As métricas e figuras são produzidas diretamente pelos resultados calculados; não há vetores de valores digitados manualmente.

In [ ]:
output_dir = Path("/content/results_notebook") if IN_COLAB else Path("../results_notebook")
output_dir.mkdir(exist_ok=True)

write_summary(results, output_dir)
write_last_cycle_series(results, output_dir)
write_cycle_summary(results, output_dir)
plot_results(results, output_dir)

print(f"Arquivos salvos em: {output_dir.resolve()}")